In [1]:
import pandas as pd
import numpy as np
import anndata as ad
from pathlib import Path
from tqdm import tqdm

In [2]:
def load_tcga() -> ad.AnnData:
    tcga_path = Path("/cluster/work/boeva/eheiss/datasets/TCGA")
    sample_sheet_path = tcga_path / "gdc_sample_sheet.2026-04-01.tsv"
    sample_sheet = pd.read_csv(sample_sheet_path, delimiter = "\t")
    clinical_path = tcga_path / "clinical.cohort.2026-03-30" / "clinical.tsv"
    clinical = pd.read_csv(
        clinical_path,
        delimiter="\t",
        low_memory=False,
        na_values=["--", "'--", "'--'"])
    
    sample_sheet = sample_sheet.sort_values("File ID")
    sample_sheet = sample_sheet.drop_duplicates("Sample ID", keep="first")

    sample_sheet = sample_sheet.merge(
        clinical[[
            "cases.submitter_id",
            "demographic.days_to_death",
            "diagnoses.days_to_last_follow_up",
            "demographic.vital_status",
        ]].drop_duplicates("cases.submitter_id"),
        left_on="Case ID",
        right_on="cases.submitter_id",
        how="left")

    sample_sheet["demographic.days_to_death"] = pd.to_numeric(
        sample_sheet["demographic.days_to_death"], errors="coerce")
    sample_sheet["diagnoses.days_to_last_follow_up"] = pd.to_numeric(
        sample_sheet["diagnoses.days_to_last_follow_up"], errors="coerce")
    sample_sheet["survival_time"] = sample_sheet["demographic.days_to_death"].combine_first(
        sample_sheet["diagnoses.days_to_last_follow_up"])

    sample_sheet["event"] = (sample_sheet["demographic.vital_status"] == "Dead").astype(int)

    adatas = []

    for i in tqdm(range(len(sample_sheet))):
        case_id = sample_sheet["Case ID"].iloc[i]
        sample_id = sample_sheet["Sample ID"].iloc[i]
        file_id = sample_sheet["File ID"].iloc[i]
        file_name = sample_sheet["File Name"].iloc[i]
        project_id = sample_sheet["Project ID"].iloc[i]
        tissue_type = sample_sheet["Tissue Type"].iloc[i]
        survival_time = sample_sheet["survival_time"].iloc[i]
        event = sample_sheet["event"].iloc[i]

        file_path = tcga_path / "raw_data" / file_id / file_name
        df = pd.read_csv(file_path, sep="\t", comment = "#")

        # remove non-gene rows and get rid of version
        df = df[~df["gene_id"].str.startswith("N_")]
        df["gene_id"] = df["gene_id"].str.split(".").str[0]
        
        df = (
            df.groupby("gene_id", as_index=False)
              .agg({
                  "unstranded": "sum",
                  "gene_name": "first",
                  "gene_type": "first",
              }))
        

        counts = df["unstranded"].values.reshape(1, -1)

        adata_sample = ad.AnnData(
            X=counts,
            obs = pd.DataFrame(
                {
                    "case_id": [case_id],
                    "sample_id": sample_id,
                    "project_id": [project_id],
                    "tissue_type": [tissue_type],
                    "survival_time": [survival_time],
                    "event": [event]
                },
                index=[sample_id]
            ),
            var=pd.DataFrame(
                {
                    "gene_id": df["gene_id"].values,
                    "gene_name": df["gene_name"].values,
                },
                index=df["gene_id"].values)
        )

        adatas.append(adata_sample)

    adata = ad.concat(adatas, join="inner")
    
    adata.write("/cluster/work/boeva/eheiss/datasets/TCGA/tcga.h5ad")

    return adata

In [3]:
tcga = load_tcga()

  6%|▌         | 634/11370 [03:17<55:45,  3.21it/s]  


KeyboardInterrupt: 